# Thesis Python Analysis and Modelling Pipeline

This notebook contains the Python part of the thesis pipeline. The SQL-to-Python data transfer is intentionally omitted. The notebook assumes that `df` is already loaded from `employee_week_feature_dataset`, and `df_pv` is already loaded from `financial_data`.

The notebook is structured for repository review: code cells are cleared of outputs, long reporting logic is split into sections, and obvious notebook leftovers were removed without changing the analytical pipeline.

## Setup and technical date trimming

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# The notebook assumes that df is already loaded from employee_week_feature_dataset.
df["business_wk"] = pd.to_datetime(df["business_wk"], errors="coerce")

max_week = df["business_wk"].max()
analysis_end_week = max_week - pd.Timedelta(weeks=2)

print("Original df shape:", df.shape)
print("Max week in data:", max_week)
print("Analysis end week:", analysis_end_week)

df = df[df["business_wk"] <= analysis_end_week].copy()

print("Filtered df shape:", df.shape)
print("New max business_wk:", df["business_wk"].max())

wk_stats_tmp = (
    df.groupby("business_wk")
    .agg(
        employees_cnt=("delivery_agent_rk", "nunique"),
        rows_cnt=("delivery_agent_rk", "size"),
    )
    .reset_index()
    .sort_values("business_wk")
)

employees_median = wk_stats_tmp["employees_cnt"].median()

stable_weeks = wk_stats_tmp[
    wk_stats_tmp["employees_cnt"] >= employees_median * 0.5
]["business_wk"]

max_stable_week = stable_weeks.max()

print("Original df shape:", df.shape)
print("Employees median per week:", employees_median)
print("Max stable week:", max_stable_week)

df = df[df["business_wk"] <= max_stable_week].copy()

print("Filtered df shape:", df.shape)
print("shape:", df.shape)
print("\ncolumns:")
print(df.columns.tolist())
print("\ndtypes:")
print(df.dtypes)


## Duplicate checks

In [ ]:
dup_cnt = df.duplicated(subset=["delivery_agent_rk", "business_wk"]).sum()
print("duplicates by (delivery_agent_rk, business_wk):", dup_cnt)

if dup_cnt > 0:
    print("duplicates by (delivery_agent_rk, business_wk):")
    print(df[df.duplicated(subset=["delivery_agent_rk", "business_wk"])])

## Missing values

In [ ]:
na_stats = pd.DataFrame({
    "na_cnt": df.isna().sum(),
    "na_share": df.isna().mean().round(4)
}).sort_values(["na_share", "na_cnt"], ascending=False)

na_stats.head(50)

## Missing-value chart

In [ ]:
top_na = na_stats.head(20)

plt.figure(figsize=(10, 6))
plt.barh(top_na.index[::-1], top_na["na_share"][::-1])
plt.title("Top-20 columns by missing share")
plt.xlabel("NA share")
plt.tight_layout()
plt.show()

## Base descriptive statistics

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

desc = df[num_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

desc["feature"] = desc.index
desc["skew"] = df[num_cols].skew(numeric_only=True)
desc["kurtosis"] = df[num_cols].kurtosis(numeric_only=True)
desc = desc.reset_index(drop=True)

print(desc.sort_values("feature").to_string())

## Employee-week coverage

In [ ]:
print("unique employees:", df["delivery_agent_rk"].nunique())
print("unique weeks:", df["business_wk"].nunique())

emp_weeks = (
    df.groupby("delivery_agent_rk")["business_wk"]
      .nunique()
      .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
)

print("\nweeks per employee:")
print(emp_weeks)

## Weekly coverage checks

In [ ]:
wk_stats = (
    df.groupby("business_wk")
      .agg(
          employees_cnt=("delivery_agent_rk", "nunique"),
          rows_cnt=("delivery_agent_rk", "size"),
          held_meetings_amt=("held_meetings_amt", "sum"),
          scheduled_meetings_amt=("scheduled_meetings_amt", "sum"),
          qc_error_cnt=("qc_error_cnt", "sum")
      )
      .reset_index()
      .sort_values("business_wk")
)

wk_stats["employees_cnt_prev"] = wk_stats["employees_cnt"].shift(1)
wk_stats["employees_cnt_ratio_to_prev"] = (
    wk_stats["employees_cnt"] / wk_stats["employees_cnt_prev"]
)

wk_stats.tail(15)

## Time-series diagnostics

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(wk_stats["business_wk"], wk_stats["employees_cnt"])
plt.title("Employees per week")
plt.xlabel("business_wk")
plt.ylabel("employees_cnt")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(wk_stats["business_wk"], wk_stats["held_meetings_amt"])
plt.title("Held meetings by week")
plt.xlabel("business_wk")
plt.ylabel("held_meetings_amt")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## EDA feature list and plotting function

In [ ]:
eda_cols = [
    "meetings_amt",
    "scheduled_meetings_amt",
    "held_meetings_amt",
    "success_meetings_amt",
    "held_from_scheduled_ratio",
    "success_from_held_ratio",
    "avg_tasks_on_meeting_amt",
    "tasks_per_held_meeting_amt",
    "avg_offer_cnt_per_meeting_amt",
    "avg_offer_voice_percent",
    "avg_voice_score",
    "voiced_meeting_ratio",
    "avg_schedule_meeting_coverage_ratio",
    "avg_schedule_span_coverage_ratio",
    "held_meeting_days_cnt",
    "held_days_from_region_work_days_ratio",
    "qc_error_cnt",
    "qc_error_days_cnt",
    "qc_error_weight_sum",
    "qc_error_cnt_per_held_meeting",
    "qc_error_weight_sum_per_held_meeting"
]

eda_cols = [c for c in eda_cols if c in df.columns]
print(eda_cols)

def plot_hist_box(data, col, bins=50):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    series = data[col].dropna()

    axes[0].hist(series, bins=bins)
    axes[0].set_title(f"Histogram: {col}")
    axes[0].set_xlabel(col)

    axes[1].boxplot(series, vert=False)
    axes[1].set_title(f"Boxplot: {col}")
    axes[1].set_xlabel(col)

    plt.tight_layout()
    plt.show()

## EDA feature distributions

In [ ]:
for col in eda_cols:
    plot_hist_box(df, col, bins=40)

## IQR outlier screening

In [ ]:
outlier_stats = []

for col in eda_cols:
    s = df[col].dropna()
    if len(s) == 0:
        continue

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_stats.append({
        "column": col,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_cnt": outliers,
        "outlier_share": round(outliers / len(df), 4)
    })

outlier_stats = pd.DataFrame(outlier_stats).sort_values("outlier_share", ascending=False)

outlier_stats

## Logical consistency checks

In [ ]:
checks = {}

checks["held_gt_scheduled"] = (df["held_meetings_amt"] > df["scheduled_meetings_amt"]).sum()
checks["success_gt_held"] = (df["success_meetings_amt"] > df["held_meetings_amt"]).sum()
checks["held_ratio_gt_1"] = (df["held_from_scheduled_ratio"] > 1).sum()
checks["success_ratio_gt_1"] = (df["success_from_held_ratio"] > 1).sum()
checks["voice_percent_gt_100"] = (df["avg_offer_voice_percent"] > 100).sum()
checks["voice_percent_lt_0"] = (df["avg_offer_voice_percent"] < 0).sum()
checks["negative_meetings"] = (df["held_meetings_amt"] < 0).sum()
checks["negative_qc"] = (df["qc_error_weight_sum"] < 0).sum()

print(pd.Series(checks))

## Feature correlation matrix

In [ ]:
corr = df[eda_cols].corr()

plt.figure(figsize=(14, 10))
plt.imshow(corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

## Strong correlation pairs

In [ ]:
corr_pairs = (
    corr.where(~np.eye(corr.shape[0], dtype=bool))
        .stack()
        .reset_index()
)
corr_pairs.columns = ["feature_1", "feature_2", "corr"]
corr_pairs["abs_corr"] = corr_pairs["corr"].abs()

corr_pairs = (
    corr_pairs.sort_values("abs_corr", ascending=False)
              .drop_duplicates(subset=["abs_corr", "corr"])
)

corr_pairs.head(30)

## Candidate engagement features

In [ ]:
engagement_candidate_cols = [
    "held_from_scheduled_ratio",
    "tasks_per_held_meeting_amt",
    "avg_offer_voice_percent",
    "avg_voice_score",
    "voiced_meeting_ratio",
    "avg_schedule_meeting_coverage_ratio",
    "avg_schedule_span_coverage_ratio",
    "qc_error_cnt_per_held_meeting",
    "qc_error_weight_sum_per_held_meeting"
]

engagement_candidate_cols = [c for c in engagement_candidate_cols if c in df.columns]

cand_corr = df[engagement_candidate_cols].corr()

plt.figure(figsize=(10, 8))
plt.imshow(cand_corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(cand_corr.columns)), cand_corr.columns, rotation=90)
plt.yticks(range(len(cand_corr.index)), cand_corr.index)
plt.title("Candidate engagement features correlation")
plt.tight_layout()
plt.show()

cand_corr

## QC feature analysis

In [ ]:
qc_cols = [
    "qc_error_cnt",
    "qc_error_days_cnt",
    "qc_error_weight_sum",
    "qc_error_weight_avg",
    "qc_error_weight_max",
    "qc_error_cnt_per_held_meeting",
    "qc_error_weight_sum_per_held_meeting",
    "qc_error_days_per_held_day_ratio"
]

qc_cols = [c for c in qc_cols if c in df.columns]

df["has_qc_error_flg"] = (df["qc_error_cnt"] > 0).astype(int)

compare_cols = [
    "held_from_scheduled_ratio",
    "tasks_per_held_meeting_amt",
    "avg_offer_voice_percent",
    "avg_voice_score",
    "voiced_meeting_ratio",
    "avg_schedule_meeting_coverage_ratio"
]
compare_cols = [c for c in compare_cols if c in df.columns]

qc_compare = df.groupby("has_qc_error_flg")[compare_cols].mean().T
qc_compare

## Employee-level profile

In [ ]:
employee_profile = (
    df.groupby("delivery_agent_rk")[engagement_candidate_cols]
      .agg(["mean", "median", "std"])
)

employee_profile.head()

## Zero-meeting weeks by employee

In [ ]:
employee_zero_weeks = (
    df.assign(zero_held_week=(df["held_meetings_amt"] == 0).astype(int))
      .groupby("delivery_agent_rk")
      .agg(
          total_weeks=("business_wk", "nunique"),
          zero_held_weeks=("zero_held_week", "sum")
      )
)

employee_zero_weeks["zero_held_weeks_share"] = (
    employee_zero_weeks["zero_held_weeks"] / employee_zero_weeks["total_weeks"]
)

employee_zero_weeks.sort_values("zero_held_weeks_share", ascending=False).head(20)

## Employee-level time-series examples

In [ ]:
sample_employees = df["delivery_agent_rk"].dropna().drop_duplicates().sample(
    min(3, df["delivery_agent_rk"].nunique()),
    random_state=SEED
)

for emp in sample_employees:
    tmp = df[df["delivery_agent_rk"] == emp].sort_values("business_wk")

    plt.figure(figsize=(12, 4))
    plt.plot(tmp["business_wk"], tmp["held_from_scheduled_ratio"], label="held_from_scheduled_ratio")

    if "avg_offer_voice_percent" in tmp.columns:
        plt.plot(tmp["business_wk"], tmp["avg_offer_voice_percent"] / 100.0, label="avg_offer_voice_percent / 100")

    if "qc_error_weight_sum_per_held_meeting" in tmp.columns:
        plt.plot(tmp["business_wk"], tmp["qc_error_weight_sum_per_held_meeting"], label="qc_error_weight_sum_per_held_meeting")

    plt.title(f"Employee {emp}")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Draft engagement index

In [ ]:
df_idx = df.copy()

base_pos_cols = [
    "held_from_scheduled_ratio",
    "tasks_per_held_meeting_amt",
]

voice_cols = [
    "avg_offer_voice_percent",
    "avg_voice_score",
    "voiced_meeting_ratio",
]

neg_cols = [
    "qc_error_weight_sum_per_held_meeting",
]

all_cols = base_pos_cols + voice_cols + neg_cols
all_cols = [col for col in all_cols if col in df_idx.columns]
base_pos_cols = [col for col in base_pos_cols if col in df_idx.columns]
voice_cols = [col for col in voice_cols if col in df_idx.columns]
neg_cols = [col for col in neg_cols if col in df_idx.columns]

print("base_pos_cols:", base_pos_cols)
print("voice_cols:", voice_cols)
print("neg_cols:", neg_cols)

for col in all_cols:
    df_idx[col] = pd.to_numeric(df_idx[col], errors="coerce")

df_idx[all_cols] = df_idx[all_cols].replace([np.inf, -np.inf], np.nan)

if "held_from_scheduled_ratio" in df_idx.columns:
    df_idx["held_from_scheduled_ratio"] = df_idx["held_from_scheduled_ratio"].clip(0, 1)

if "avg_offer_voice_percent" in df_idx.columns:
    df_idx["avg_offer_voice_percent"] = df_idx["avg_offer_voice_percent"].clip(0, 100)

if "avg_voice_score" in df_idx.columns:
    df_idx["avg_voice_score"] = df_idx["avg_voice_score"].clip(0, 1)

if "voiced_meeting_ratio" in df_idx.columns:
    df_idx["voiced_meeting_ratio"] = df_idx["voiced_meeting_ratio"].clip(0, 1)

tail_cols = [
    "tasks_per_held_meeting_amt",
    "qc_error_weight_sum_per_held_meeting",
]

for col in tail_cols:
    if col in df_idx.columns:
        p99 = df_idx[col].quantile(0.99)
        df_idx[col] = df_idx[col].clip(lower=0, upper=p99)
        print(f"{col}: clipped by p99 =", p99)

if "qc_error_weight_sum_per_held_meeting" in df_idx.columns:
    df_idx["qc_error_weight_sum_per_held_meeting"] = (
        df_idx["qc_error_weight_sum_per_held_meeting"].fillna(0)
    )

for col in base_pos_cols + voice_cols:
    df_idx[col] = df_idx[col].fillna(df_idx[col].median())

for col in all_cols:
    mean_ = df_idx[col].mean()
    std_ = df_idx[col].std()

    if pd.notna(std_) and std_ > 0:
        df_idx[f"{col}_z"] = (df_idx[col] - mean_) / std_
    else:
        df_idx[f"{col}_z"] = 0.0

voice_z_cols = [f"{col}_z" for col in voice_cols if f"{col}_z" in df_idx.columns]
df_idx["voice_component_z"] = df_idx[voice_z_cols].mean(axis=1)

positive_components = []

if "held_from_scheduled_ratio_z" in df_idx.columns:
    positive_components.append("held_from_scheduled_ratio_z")

if "tasks_per_held_meeting_amt_z" in df_idx.columns:
    positive_components.append("tasks_per_held_meeting_amt_z")

positive_components.append("voice_component_z")

negative_components = [
    "qc_error_weight_sum_per_held_meeting_z",
]
negative_components = [col for col in negative_components if col in df_idx.columns]

df_idx["engagement_index_draft"] = (
    df_idx[positive_components].mean(axis=1)
    - df_idx[negative_components].mean(axis=1)
)

check_cols = positive_components + negative_components + ["engagement_index_draft"]

for col in check_cols:
    print(
        col,
        "| na =", df_idx[col].isna().sum(),
        "| inf =", np.isinf(df_idx[col]).sum(),
        "| mean =", round(df_idx[col].mean(), 6),
        "| std =", round(df_idx[col].std(), 6),
    )


## Engagement index summary

In [ ]:
num_cols = df_idx.select_dtypes(include=[np.number]).columns.tolist()

eda_summary = pd.DataFrame(index=df_idx.columns)
eda_summary["dtype"] = df_idx.dtypes.astype(str)
eda_summary["na_share"] = df_idx.isna().mean()
eda_summary["n_unique"] = df_idx.nunique()

for col in ["mean", "std", "min", "p50", "p95", "max"]:
    eda_summary[col] = np.nan

eda_summary.loc[num_cols, "mean"] = df_idx[num_cols].mean()
eda_summary.loc[num_cols, "std"] = df_idx[num_cols].std()
eda_summary.loc[num_cols, "min"] = df_idx[num_cols].min()
eda_summary.loc[num_cols, "p50"] = df_idx[num_cols].median()
eda_summary.loc[num_cols, "p95"] = df_idx[num_cols].quantile(0.95)
eda_summary.loc[num_cols, "max"] = df_idx[num_cols].max()

print(eda_summary.sort_values(["na_share", "dtype"], ascending=[False, True]).to_string())

## Engagement quantiles

In [ ]:
df_idx = df_idx.copy()

df_idx["engagement_group_3"] = pd.qcut(
    df_idx["engagement_index_draft"],
    q=3,
    labels=["low", "mid", "high"]
)

summary_cols = [
    "held_from_scheduled_ratio",
    "tasks_per_held_meeting_amt",
    "avg_offer_voice_percent",
    "avg_voice_score",
    "voiced_meeting_ratio",
    "voice_component_z",
    "qc_error_weight_sum_per_held_meeting",
    "engagement_index_draft"
]

summary_cols = [c for c in summary_cols if c in df_idx.columns]

group_summary = (
    df_idx.groupby("engagement_group_3")[summary_cols]
    .agg(["mean", "median"])
)

print(group_summary.to_string())

plt.figure(figsize=(10, 4))
plt.hist(df_idx["engagement_index_draft"], bins=50)
plt.title("Engagement index draft distribution")
plt.xlabel("engagement_index_draft")
plt.ylabel("count")
plt.tight_layout()
plt.show()

## Engagement index distribution

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.hist(df_idx["engagement_index_draft"], bins=50)
plt.title("Engagement index draft distribution")
plt.xlabel("engagement_index_draft")
plt.ylabel("count")
plt.tight_layout()
plt.show()

## Engagement index checks

In [ ]:
check = (
    df_idx.groupby("engagement_group_3")[
        [
            "held_from_scheduled_ratio",
            "tasks_per_held_meeting_amt",
            "avg_offer_voice_percent",
            "avg_voice_score",
            "voiced_meeting_ratio",
            "qc_error_weight_sum_per_held_meeting"
        ]
    ]
    .mean()
)

print(check.to_string())

## Dataset output check

In [ ]:
df_idx.info()


## Financial data merge

In [ ]:
pv_cols = [
    "pv",
    "pv_50",
    "pv_for_pi",
    "pv_50_for_pi",
    "total_cost",
    "total_cost_for_pi"
]

df_idx = df_idx.copy()
df_pv = df_pv.copy()

df_idx["business_wk"] = pd.to_datetime(df_idx["business_wk"], errors="coerce")
df_pv["business_wk"] = pd.to_datetime(df_pv["business_wk"], errors="coerce")

df_model = df_idx.merge(
    df_pv,
    on=["delivery_agent_rk", "business_wk"],
    how="left"
)

df_model = df_model.copy()
df_model["business_wk"] = pd.to_datetime(df_model["business_wk"], errors="coerce")

for col in pv_cols:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce").fillna(0)

df_model = df_model.sort_values(["delivery_agent_rk", "business_wk"]).copy()

df_model["next_business_wk"] = (
    df_model.groupby("delivery_agent_rk")["business_wk"].shift(-1)
)

df_model["week_gap_days"] = (
    df_model["next_business_wk"] - df_model["business_wk"]
).dt.days

print("PV week gap distribution:")
print(df_model["week_gap_days"].value_counts(dropna=False).sort_index())

for col in pv_cols:
    for suffix in ["_next_wk", "_next_wk_raw"]:
        old_col = f"{col}{suffix}"
        if old_col in df_model.columns:
            df_model = df_model.drop(columns=[old_col])

for col in pv_cols:
    if col in df_model.columns:
        df_model[f"{col}_next_wk_raw"] = (
            df_model.groupby("delivery_agent_rk")[col].shift(-1)
        )

        df_model[f"{col}_next_wk"] = np.where(
            df_model["week_gap_days"] == 7,
            df_model[f"{col}_next_wk_raw"],
            np.nan
        )

df_model["pv_next_wk_log1p"] = np.log1p(
    pd.to_numeric(df_model["pv_next_wk"], errors="coerce").clip(lower=0)
)

df_model["pv_50_next_wk_log1p"] = np.log1p(
    pd.to_numeric(df_model["pv_50_next_wk"], errors="coerce").clip(lower=0)
)

print("\nRows in df_model:", len(df_model))

print("\nRows with gap == 7:")
print((df_model["week_gap_days"] == 7).sum())

print("\nNon-null pv_next_wk:")
print(df_model["pv_next_wk"].notna().sum())

bad_next_pv = df_model[
    df_model["pv_next_wk"].notna()
    & (df_model["week_gap_days"] != 7)
]

miss_next_pv = df_model[
    (df_model["week_gap_days"] == 7)
    & df_model["pv_next_wk"].isna()
]

print("\nRows where pv_next_wk is not null but gap != 7:")
print(len(bad_next_pv))

print("\nRows where gap == 7 but pv_next_wk is null:")
print(len(miss_next_pv))

df_model["engagement_group_3"] = pd.qcut(
    df_model["engagement_index_draft"],
    q=3,
    labels=["low", "mid", "high"]
)

df_model["engagement_group_5"] = pd.qcut(
    df_model["engagement_index_draft"],
    q=5,
    labels=["q1", "q2", "q3", "q4", "q5"]
)

print("\nPV mean by engagement_group_3:")
print(
    df_model.groupby("engagement_group_3")[["pv_next_wk", "pv_50_next_wk"]]
    .mean()
    .to_string()
)

print("\nPV mean by engagement_group_5:")
print(
    df_model.groupby("engagement_group_5")[["pv_next_wk", "pv_50_next_wk"]]
    .mean()
    .to_string()
)

print("\nPV correlations:")
print(
    df_model[
        [
            "engagement_index_draft",
            "pv_next_wk",
            "pv_50_next_wk",
            "pv_next_wk_log1p",
            "pv_50_next_wk_log1p"
        ]
    ]
    .corr()
    .round(6)
    .to_string()
)

print("\nPV Spearman correlations:")
print(
    df_model[
        [
            "engagement_index_draft",
            "pv_next_wk",
            "pv_50_next_wk",
            "pv_next_wk_log1p",
            "pv_50_next_wk_log1p"
        ]
    ]
    .corr(method="spearman")
    .round(6)
    .to_string()
)

## Financial correlation check

In [ ]:
print(
    df_model[["engagement_index_draft", "pv_next_wk", "pv_50_next_wk", "pv_for_pi_next_wk"]]
    .corr()
    .round(3)
    .to_string()
)

## Financial quantiles

In [ ]:
df_model["engagement_group_5"] = pd.qcut(
    df_model["engagement_index_draft"],
    q=5,
    labels=["q1", "q2", "q3", "q4", "q5"]
)

print(
    df_model.groupby("engagement_group_5")[["pv_next_wk", "pv_50_next_wk"]]
    .mean()
    .to_string()
)

## Financial log transformation

In [ ]:
df_model["pv_next_wk_log1p"] = np.log1p(df_model["pv_next_wk"].clip(lower=0))
df_model["pv_50_next_wk_log1p"] = np.log1p(df_model["pv_50_next_wk"].clip(lower=0))

print(
    df_model[["engagement_index_draft", "pv_next_wk_log1p", "pv_50_next_wk_log1p"]]
    .corr()
    .round(3)
    .to_string()
)

## Spearman correlation for EI and PV

In [ ]:
display(df_model[["engagement_index_draft", "pv_next_wk_log1p", "pv_50_next_wk_log1p"]].corr(method="spearman"))

## Baseline OLS regression

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

df_reg = df_model.copy()

df_reg["pv_next_wk_log1p"] = np.log1p(df_reg["pv_next_wk"].clip(lower=0))
df_reg["pv_50_next_wk_log1p"] = np.log1p(df_reg["pv_50_next_wk"].clip(lower=0))

for col in [
    "engagement_index_draft",
    "held_meetings_amt",
    "meetings_amt",
    "meeting_regions_cnt"
]:
    if col in df_reg.columns:
        df_reg[col] = pd.to_numeric(df_reg[col], errors="coerce")

df_reg = df_reg.replace([np.inf, -np.inf], np.nan)

model_df_1 = df_reg[[
    "pv_next_wk_log1p",
    "engagement_index_draft"
]].dropna().copy()

X1 = sm.add_constant(model_df_1[["engagement_index_draft"]])
y1 = model_df_1["pv_next_wk_log1p"]

model_1 = sm.OLS(y1, X1).fit()
print(model_1.summary())

## OLS with activity controls

In [ ]:
model_df_2 = df_reg[[
    "pv_next_wk_log1p",
    "engagement_index_draft",
    "held_meetings_amt",
    "meeting_regions_cnt"
]].dropna().copy()

X2 = sm.add_constant(model_df_2[
    ["engagement_index_draft", "held_meetings_amt", "meeting_regions_cnt"]
])
y2 = model_df_2["pv_next_wk_log1p"]

model_2 = sm.OLS(y2, X2).fit()
print(model_2.summary())

## OLS with region controls

In [ ]:
df_reg["held_meetings_amt_log1p"] = np.log1p(df_reg["held_meetings_amt"].clip(lower=0))

model_df_3 = df_reg[[
    "pv_next_wk_log1p",
    "engagement_index_draft",
    "held_meetings_amt_log1p",
    "meeting_regions_cnt"
]].dropna().copy()

X3 = sm.add_constant(model_df_3[
    ["engagement_index_draft", "held_meetings_amt_log1p", "meeting_regions_cnt"]
])
y3 = model_df_3["pv_next_wk_log1p"]

model_3 = sm.OLS(y3, X3).fit()
print(model_3.summary())

model_df_base = model_df_3.copy()

X_base = sm.add_constant(model_df_base[
    ["held_meetings_amt_log1p", "meeting_regions_cnt"]
])

y_base = model_df_base["pv_next_wk_log1p"]

model_base = sm.OLS(y_base, X_base).fit()

model_base_cluster = sm.OLS(y_base, X_base).fit(
    cov_type="cluster",
    cov_kwds={"groups": model_df_base["delivery_agent_rk"]}
)

## Cluster-robust OLS

In [ ]:
model_df_3_cluster = df_reg[[
    "delivery_agent_rk",
    "pv_next_wk_log1p",
    "engagement_index_draft",
    "held_meetings_amt_log1p",
    "meeting_regions_cnt"
]].dropna().copy()

X3 = sm.add_constant(model_df_3_cluster[
    ["engagement_index_draft", "held_meetings_amt_log1p", "meeting_regions_cnt"]
])
y3 = model_df_3_cluster["pv_next_wk_log1p"]

model_3_cluster = sm.OLS(y3, X3).fit(
    cov_type="cluster",
    cov_kwds={"groups": model_df_3_cluster["delivery_agent_rk"]}
)

print(model_3_cluster.summary())

## OLS model comparison

In [ ]:
model_df_4 = df_reg[[
    "pv_50_next_wk_log1p",
    "engagement_index_draft",
    "held_meetings_amt_log1p",
    "meeting_regions_cnt"
]].dropna().copy()

X4 = sm.add_constant(model_df_4[
    ["engagement_index_draft", "held_meetings_amt_log1p", "meeting_regions_cnt"]
])
y4 = model_df_4["pv_50_next_wk_log1p"]

model_4 = sm.OLS(y4, X4).fit()
print(model_4.summary())

## Regression summary tables

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

df_reg = df_model.copy()

numeric_cols_for_reg = [
    "engagement_index_draft",
    "held_meetings_amt",
    "meetings_amt",
    "meeting_regions_cnt",
    "pv_next_wk",
    "pv_50_next_wk"
]

for col in numeric_cols_for_reg:
    if col in df_reg.columns:
        df_reg[col] = pd.to_numeric(df_reg[col], errors="coerce")

df_reg = df_reg.replace([np.inf, -np.inf], np.nan)

df_reg["pv_next_wk_log1p"] = np.log1p(
    df_reg["pv_next_wk"].clip(lower=0)
)

df_reg["pv_50_next_wk_log1p"] = np.log1p(
    df_reg["pv_50_next_wk"].clip(lower=0)
)

df_reg["held_meetings_amt_log1p"] = np.log1p(
    df_reg["held_meetings_amt"].clip(lower=0)
)

def coef_table(model):
    out = pd.DataFrame({
        "coef": model.params,
        "std_err": model.bse,
        "p_value": model.pvalues,
        "t_or_z": model.tvalues
    })
    return out

def print_model_result(name, model):
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)
    print(coef_table(model).to_string())
    print("R2:", model.rsquared)
    print("Adj R2:", model.rsquared_adj)
    print("NOBS:", model.nobs)

model_df_1 = df_reg[[
    "delivery_agent_rk",
    "pv_next_wk_log1p",
    "engagement_index_draft"
]].dropna().copy()

X1 = sm.add_constant(model_df_1[[
    "engagement_index_draft"
]])

y1 = model_df_1["pv_next_wk_log1p"]

model_1 = sm.OLS(y1, X1).fit()

model_df_2 = df_reg[[
    "delivery_agent_rk",
    "pv_next_wk_log1p",
    "engagement_index_draft",
    "held_meetings_amt",
    "meeting_regions_cnt"
]].dropna().copy()

X2 = sm.add_constant(model_df_2[[
    "engagement_index_draft",
    "held_meetings_amt",
    "meeting_regions_cnt"
]])

y2 = model_df_2["pv_next_wk_log1p"]

model_2 = sm.OLS(y2, X2).fit()

model_df_3 = df_reg[[
    "delivery_agent_rk",
    "pv_next_wk_log1p",
    "engagement_index_draft",
    "held_meetings_amt_log1p",
    "meeting_regions_cnt"
]].dropna().copy()

X3 = sm.add_constant(model_df_3[[
    "engagement_index_draft",
    "held_meetings_amt_log1p",
    "meeting_regions_cnt"
]])

y3 = model_df_3["pv_next_wk_log1p"]

model_3 = sm.OLS(y3, X3).fit()

model_3_cluster = sm.OLS(y3, X3).fit(
    cov_type="cluster",
    cov_kwds={"groups": model_df_3["delivery_agent_rk"]}
)

model_df_4 = df_reg[[
    "delivery_agent_rk",
    "pv_50_next_wk_log1p",
    "engagement_index_draft",
    "held_meetings_amt_log1p",
    "meeting_regions_cnt"
]].dropna().copy()

X4 = sm.add_constant(model_df_4[[
    "engagement_index_draft",
    "held_meetings_amt_log1p",
    "meeting_regions_cnt"
]])

y4 = model_df_4["pv_50_next_wk_log1p"]

model_4 = sm.OLS(y4, X4).fit()

X_base = sm.add_constant(model_df_3[[
    "held_meetings_amt_log1p",
    "meeting_regions_cnt"
]])

y_base = model_df_3["pv_next_wk_log1p"]

model_base = sm.OLS(y_base, X_base).fit()

model_base_cluster = sm.OLS(y_base, X_base).fit(
    cov_type="cluster",
    cov_kwds={"groups": model_df_3["delivery_agent_rk"]}
)

print_model_result("MODEL 1: PV ~ EI", model_1)
print_model_result("MODEL 2: PV ~ EI + held meetings + regions", model_2)
print_model_result("MODEL 3: PV ~ EI + log held meetings + regions", model_3)
print_model_result("MODEL 3 CLUSTER: cluster-robust SE by employee", model_3_cluster)
print_model_result("MODEL 4: PV_50 ~ EI + log held meetings + regions", model_4)
print_model_result("BASE MODEL: PV ~ log held meetings + regions", model_base)
print_model_result("BASE MODEL CLUSTER: cluster-robust SE by employee", model_base_cluster)

print("\n" + "=" * 100)
print("DELTA R2: MODEL 3 VS BASE MODEL")
print("=" * 100)
print("BASE MODEL R2:", model_base.rsquared)
print("MODEL 3 R2:", model_3.rsquared)
print("DELTA R2:", model_3.rsquared - model_base.rsquared)

print("\n" + "=" * 100)
print("SAMPLE SIZE CONTROL")
print("=" * 100)
print("model_1 nobs:", model_1.nobs)
print("model_2 nobs:", model_2.nobs)
print("model_3 nobs:", model_3.nobs)
print("model_4 nobs:", model_4.nobs)
print("model_base nobs:", model_base.nobs)

print("\nExpected strict next-week sample size:")
print("rows with gap == 7:", (df_model["week_gap_days"] == 7).sum())
print("non-null pv_next_wk:", df_model["pv_next_wk"].notna().sum())

## EI coefficient extraction

In [ ]:
def coef_table(model):
    out = pd.DataFrame({
        "coef": model.params,
        "std_err": model.bse,
        "p_value": model.pvalues
    })
    out["t_or_z"] = model.tvalues
    return out

print("MODEL 1")
print(coef_table(model_1).to_string())

print("\nMODEL 2")
print(coef_table(model_2).to_string())

print("\nMODEL 3")
print(coef_table(model_3).to_string())

print("\nMODEL 4")
print(coef_table(model_4).to_string())

## R-squared extraction

In [ ]:
print("MODEL 1 R2:", model_1.rsquared)
print("MODEL 2 R2:", model_2.rsquared)
print("MODEL 3 R2:", model_3.rsquared)
print("MODEL 4 R2:", model_4.rsquared)

## Incremental explanatory power

In [ ]:
X_base = sm.add_constant(model_df_3[
    ["held_meetings_amt_log1p", "meeting_regions_cnt"]
])
y_base = model_df_3["pv_next_wk_log1p"]

model_base = sm.OLS(y_base, X_base).fit()

print("BASE MODEL R2:", model_base.rsquared)
print("MODEL 3 R2:", model_3.rsquared)
print("DELTA R2:", model_3.rsquared - model_base.rsquared)

print("\nBASE MODEL COEFS:")
print(model_base.params)

## EI and PV relation chart

In [ ]:
import numpy as np
import pandas as pd

df_seg = df_model.copy()

for col in [
    "delivery_agent_rk",
    "business_wk",
    "engagement_index_draft",
    "pv_next_wk",
    "pv_50_next_wk"
]:
    if col in df_seg.columns:
        if col != "business_wk":
            df_seg[col] = pd.to_numeric(df_seg[col], errors="coerce")

df_seg["business_wk"] = pd.to_datetime(df_seg["business_wk"], errors="coerce")

df_seg = df_seg.replace([np.inf, -np.inf], np.nan)
df_seg = df_seg.dropna(subset=["delivery_agent_rk", "business_wk", "engagement_index_draft", "pv_next_wk"]).copy()

print(df_seg.shape)

## EI group boundaries for PV

In [ ]:
ei_cut = df_seg["engagement_index_draft"].median()
pv_cut = df_seg["pv_next_wk"].median()

print("EI median cutoff:", ei_cut)
print("PV median cutoff:", pv_cut)

ei_q40 = df_seg["engagement_index_draft"].quantile(0.4)
ei_q60 = df_seg["engagement_index_draft"].quantile(0.6)

pv_q40 = df_seg["pv_next_wk"].quantile(0.4)
pv_q60 = df_seg["pv_next_wk"].quantile(0.6)

print(ei_q40, ei_q60, pv_q40, pv_q60)

## Segment matrix preparation

In [ ]:
print("df min week:", df["business_wk"].min())
print("df max week:", df["business_wk"].max())
print("df shape:", df.shape)

wk_check = (
    df.groupby("business_wk")
      .agg(
          employees_cnt=("delivery_agent_rk", "nunique"),
          rows_cnt=("delivery_agent_rk", "size"),
          held_meetings_amt=("held_meetings_amt", "sum"),
          scheduled_meetings_amt=("scheduled_meetings_amt", "sum"),
          qc_error_cnt=("qc_error_cnt", "sum")
      )
      .reset_index()
      .sort_values("business_wk")
)

print(wk_check.tail(10).to_string(index=False))

## EI-PV segment matrix

In [ ]:
df_seg["ei_group_2"] = np.where(
    df_seg["engagement_index_draft"] >= ei_cut,
    "high_ei",
    "low_ei"
)

df_seg["pv_group_2"] = np.where(
    df_seg["pv_next_wk"] >= pv_cut,
    "high_pv",
    "low_pv"
)

df_seg["ei_pv_segment"] = df_seg["ei_group_2"] + "__" + df_seg["pv_group_2"]

print(df_seg["ei_pv_segment"].value_counts(dropna=False).to_string())

## Segment profile summary

In [ ]:
segment_summary = (
    df_seg.groupby("ei_pv_segment")
    .agg(
        rows_cnt=("delivery_agent_rk", "size"),
        employees_cnt=("delivery_agent_rk", "nunique"),
        avg_ei=("engagement_index_draft", "mean"),
        median_ei=("engagement_index_draft", "median"),
        avg_pv_next_wk=("pv_next_wk", "mean"),
        median_pv_next_wk=("pv_next_wk", "median"),
        total_pv_next_wk=("pv_next_wk", "sum")
    )
    .sort_values("total_pv_next_wk", ascending=False)
)

print(segment_summary.to_string())

## Segment shares

In [ ]:
segment_share = (
    df_seg["ei_pv_segment"]
    .value_counts(normalize=True)
    .rename("share")
    .sort_values(ascending=False)
)

print(segment_share.to_string())

## Segment renaming

In [ ]:
segment_map = {
    "high_ei__high_pv": "High EI / High PV",
    "high_ei__low_pv":  "High EI / Low PV",
    "low_ei__high_pv":  "Low EI / High PV",
    "low_ei__low_pv":   "Low EI / Low PV"
}

segment_summary_named = segment_summary.copy()
segment_summary_named.index = segment_summary_named.index.map(segment_map)

print(segment_summary_named.to_string())

## Low-EI and high-PV segment

In [ ]:
low_ei_high_pv = df_seg[df_seg["ei_pv_segment"] == "low_ei__high_pv"].copy()

print("rows:", len(low_ei_high_pv))
print("unique employees:", low_ei_high_pv["delivery_agent_rk"].nunique())
print("avg pv_next_wk:", low_ei_high_pv["pv_next_wk"].mean())
print("total pv_next_wk:", low_ei_high_pv["pv_next_wk"].sum())

profile_cols = [
    "engagement_index_draft",
    "held_from_scheduled_ratio",
    "tasks_per_held_meeting_amt",
    "avg_offer_voice_percent",
    "avg_voice_score",
    "voiced_meeting_ratio",
    "qc_error_weight_sum_per_held_meeting",
    "held_meetings_amt"
]

profile_cols = [c for c in profile_cols if c in low_ei_high_pv.columns]

print(low_ei_high_pv[profile_cols].mean().to_string())

## Top segment profiles

In [ ]:
profile_cols = [
    "engagement_index_draft",
    "pv_next_wk",
    "held_meetings_amt",
    "held_from_scheduled_ratio",
    "tasks_per_held_meeting_amt",
    "avg_offer_voice_percent",
    "avg_voice_score",
    "voiced_meeting_ratio",
    "qc_error_weight_sum_per_held_meeting"
]

profile_cols = [c for c in profile_cols if c in df_seg.columns]

segment_profiles = (
    df_seg.groupby("ei_pv_segment")[profile_cols]
    .mean()
)

segment_profiles.index = segment_profiles.index.map(segment_map)
print(segment_profiles.to_string())

## Scenario analysis

In [ ]:
beta_ei = model_3.params["engagement_index_draft"]

def relative_pv_change(delta_ei, beta=beta_ei):
    return np.exp(beta * delta_ei) - 1

scenario_grid = pd.DataFrame({
    "delta_ei": [-1.0, -0.5, -0.25, 0.25, 0.5, 1.0]
})

scenario_grid["expected_relative_change"] = (
    scenario_grid["delta_ei"].apply(relative_pv_change)
)

scenario_grid["expected_relative_change_pct"] = (
    scenario_grid["expected_relative_change"] * 100
)

print("beta_ei:", beta_ei)
print("\nScenario grid:")
print(scenario_grid.to_string(index=False))

total_observed_pv = df_seg["pv_next_wk"].sum()
avg_observed_pv_per_row = df_seg["pv_next_wk"].mean()

print("\ntotal_observed_pv:", total_observed_pv)
print("avg_observed_pv_per_row:", avg_observed_pv_per_row)

delta_ei = -0.5
rel_change = relative_pv_change(delta_ei)

expected_observed_pv_after = total_observed_pv * (1 + rel_change)
expected_observed_pv_difference = total_observed_pv - expected_observed_pv_after

print("\nWhole observed sample scenario")
print("delta_ei:", delta_ei)
print("relative change:", rel_change)
print("expected_observed_pv_after:", expected_observed_pv_after)
print("expected_observed_pv_difference:", expected_observed_pv_difference)

seg_low_ei_high_pv = df_seg[
    df_seg["ei_pv_segment"] == "low_ei__high_pv"
].copy()

seg_low_ei_high_pv_total_pv = seg_low_ei_high_pv["pv_next_wk"].sum()

delta_ei = -0.5
rel_change = relative_pv_change(delta_ei)

seg_low_ei_high_pv_expected_difference = (
    seg_low_ei_high_pv_total_pv
    - seg_low_ei_high_pv_total_pv * (1 + rel_change)
)

print("\nLow EI / High PV segment scenario")
print("segment rows:", len(seg_low_ei_high_pv))
print("segment employees:", seg_low_ei_high_pv["delivery_agent_rk"].nunique())
print("segment total pv:", seg_low_ei_high_pv_total_pv)
print("delta_ei:", delta_ei)
print("expected_pv_difference:", seg_low_ei_high_pv_expected_difference)

seg_high_ei_high_pv = df_seg[
    df_seg["ei_pv_segment"] == "high_ei__high_pv"
].copy()

seg_high_ei_high_pv_total_pv = seg_high_ei_high_pv["pv_next_wk"].sum()

delta_ei = -0.5
rel_change = relative_pv_change(delta_ei)

seg_high_ei_high_pv_expected_difference = (
    seg_high_ei_high_pv_total_pv
    - seg_high_ei_high_pv_total_pv * (1 + rel_change)
)

print("\nHigh EI / High PV segment scenario")
print("segment rows:", len(seg_high_ei_high_pv))
print("segment employees:", seg_high_ei_high_pv["delivery_agent_rk"].nunique())
print("segment total pv:", seg_high_ei_high_pv_total_pv)
print("delta_ei:", delta_ei)
print("expected_pv_difference:", seg_high_ei_high_pv_expected_difference)

top_pv_employees = (
    df_seg.groupby("delivery_agent_rk")["pv_next_wk"]
    .sum()
    .sort_values(ascending=False)
)

top_10pct_n = max(1, int(len(top_pv_employees) * 0.10))
top_10pct_ids = top_pv_employees.head(top_10pct_n).index

seg_10pct = df_seg[
    df_seg["delivery_agent_rk"].isin(top_10pct_ids)
].copy()

seg_10pct_total_pv = seg_10pct["pv_next_wk"].sum()

delta_ei = -1.0
rel_change = relative_pv_change(delta_ei)

seg_10pct_expected_difference = (
    seg_10pct_total_pv
    - seg_10pct_total_pv * (1 + rel_change)
)

print("\nTop 10% employees by observed PV scenario")
print("top 10% employees count:", len(top_10pct_ids))
print("top 10% employee-week rows:", len(seg_10pct))
print("top 10% observed PV:", seg_10pct_total_pv)
print("delta_ei:", delta_ei)
print("expected_pv_difference:", seg_10pct_expected_difference)

## Scenario result interpretation

In [ ]:
ei_q30 = df_seg["engagement_index_draft"].quantile(0.3)
ei_q70 = df_seg["engagement_index_draft"].quantile(0.7)

pv_q30 = df_seg["pv_next_wk"].quantile(0.3)
pv_q70 = df_seg["pv_next_wk"].quantile(0.7)

df_seg_extreme = df_seg[
    (
        (df_seg["engagement_index_draft"] <= ei_q30) |
        (df_seg["engagement_index_draft"] >= ei_q70)
    ) &
    (
        (df_seg["pv_next_wk"] <= pv_q30) |
        (df_seg["pv_next_wk"] >= pv_q70)
    )
].copy()

df_seg_extreme["ei_group_2"] = np.where(df_seg_extreme["engagement_index_draft"] >= ei_q70, "high_ei", "low_ei")
df_seg_extreme["pv_group_2"] = np.where(df_seg_extreme["pv_next_wk"] >= pv_q70, "high_pv", "low_pv")
df_seg_extreme["ei_pv_segment"] = df_seg_extreme["ei_group_2"] + "__" + df_seg_extreme["pv_group_2"]

print(df_seg_extreme["ei_pv_segment"].value_counts().to_string())

## ML feature engineering

In [ ]:
import numpy as np
import pandas as pd

df_ml = df_idx.copy()

df_ml["business_wk"] = pd.to_datetime(df_ml["business_wk"], errors="coerce")
df_ml = df_ml.dropna(
    subset=["delivery_agent_rk", "business_wk", "engagement_index_draft"]
).copy()

df_ml = df_ml.sort_values(["delivery_agent_rk", "business_wk"]).copy()

df_ml["next_business_wk"] = df_ml.groupby("delivery_agent_rk")["business_wk"].shift(-1)

df_ml["week_gap_days"] = (
    df_ml["next_business_wk"] - df_ml["business_wk"]
).dt.days

print("Week gap distribution:")
print(df_ml["week_gap_days"].value_counts(dropna=False).sort_index())

base_feature_cols = [
    "held_from_scheduled_ratio",
    "tasks_per_held_meeting_amt",
    "avg_offer_voice_percent",
    "avg_voice_score",
    "voiced_meeting_ratio",
    "qc_error_weight_sum_per_held_meeting",
    "held_meetings_amt",
    "meeting_regions_cnt",
]

for col in base_feature_cols:
    if col in df_ml.columns:
        df_ml[col] = pd.to_numeric(df_ml[col], errors="coerce")

df_ml["target_ei_next"] = df_ml.groupby("delivery_agent_rk")[
    "engagement_index_draft"
].shift(-1)

df_ml["target_ei_next"] = np.where(
    df_ml["week_gap_days"] == 7,
    df_ml["target_ei_next"],
    np.nan,
)

for lag in [1, 2, 4]:
    df_ml[f"ei_lag_{lag}"] = df_ml.groupby("delivery_agent_rk")[
        "engagement_index_draft"
    ].shift(lag)

lag_feature_cols = [
    "held_from_scheduled_ratio",
    "tasks_per_held_meeting_amt",
    "avg_offer_voice_percent",
    "avg_voice_score",
    "voiced_meeting_ratio",
    "qc_error_weight_sum_per_held_meeting",
    "held_meetings_amt",
]

for col in lag_feature_cols:
    df_ml[f"{col}_lag_1"] = df_ml.groupby("delivery_agent_rk")[col].shift(1)

df_ml["ei_roll_mean_4"] = df_ml.groupby("delivery_agent_rk")[
    "engagement_index_draft"
].transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())

df_ml["ei_roll_std_4"] = df_ml.groupby("delivery_agent_rk")[
    "engagement_index_draft"
].transform(lambda s: s.shift(1).rolling(4, min_periods=2).std())

df_ml["held_meetings_roll_mean_4"] = df_ml.groupby("delivery_agent_rk")[
    "held_meetings_amt"
].transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())

df_ml["ei_delta_1"] = df_ml["engagement_index_draft"] - df_ml["ei_lag_1"]
df_ml["avg_offer_voice_percent_delta_1"] = (
    df_ml["avg_offer_voice_percent"]
    - df_ml.groupby("delivery_agent_rk")["avg_offer_voice_percent"].shift(1)
)
df_ml["avg_voice_score_delta_1"] = (
    df_ml["avg_voice_score"]
    - df_ml.groupby("delivery_agent_rk")["avg_voice_score"].shift(1)
)

drop_threshold = -0.5

df_ml["ei_next_delta"] = df_ml["target_ei_next"] - df_ml["engagement_index_draft"]

df_ml["target_ei_drop_flag"] = np.where(
    df_ml["target_ei_next"].notna(),
    (df_ml["ei_next_delta"] < drop_threshold).astype(int),
    np.nan,
)

print(df_ml.shape)
print(
    df_ml[
        [
            "delivery_agent_rk",
            "business_wk",
            "engagement_index_draft",
            "target_ei_next",
            "ei_lag_1",
            "ei_lag_2",
            "ei_lag_4",
            "ei_roll_mean_4",
            "ei_roll_std_4",
            "target_ei_drop_flag",
        ]
    ]
    .head(20)
    .to_string()
)


## Time-based train-validation-test split

In [ ]:
df_ml_model = df_ml.copy()

feature_cols = [
    "engagement_index_draft",
    "held_from_scheduled_ratio",
    "tasks_per_held_meeting_amt",
    "avg_offer_voice_percent",
    "avg_voice_score",
    "voiced_meeting_ratio",
    "qc_error_weight_sum_per_held_meeting",
    "held_meetings_amt",
    "meeting_regions_cnt",
    "ei_lag_1",
    "ei_lag_2",
    "ei_lag_4",
    "ei_roll_mean_4",
    "ei_roll_std_4",
    "held_meetings_roll_mean_4",
    "ei_delta_1",
    "avg_offer_voice_percent_delta_1",
    "avg_voice_score_delta_1",
]

feature_cols = [col for col in feature_cols if col in df_ml_model.columns]

df_reg_ml = df_ml_model.dropna(subset=["target_ei_next"]).copy()

for col in feature_cols:
    df_reg_ml[col] = pd.to_numeric(df_reg_ml[col], errors="coerce")
    df_reg_ml[col] = df_reg_ml[col].replace([np.inf, -np.inf], np.nan)

train_end = df_reg_ml["business_wk"].quantile(0.70)
valid_end = df_reg_ml["business_wk"].quantile(0.85)

train_mask = df_reg_ml["business_wk"] <= train_end
valid_mask = (df_reg_ml["business_wk"] > train_end) & (
    df_reg_ml["business_wk"] <= valid_end
)
test_mask = df_reg_ml["business_wk"] > valid_end

X_train = df_reg_ml.loc[train_mask, feature_cols].copy()
y_train = df_reg_ml.loc[train_mask, "target_ei_next"].copy()

X_valid = df_reg_ml.loc[valid_mask, feature_cols].copy()
y_valid = df_reg_ml.loc[valid_mask, "target_ei_next"].copy()

X_test = df_reg_ml.loc[test_mask, feature_cols].copy()
y_test = df_reg_ml.loc[test_mask, "target_ei_next"].copy()

train_medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_medians)
X_valid = X_valid.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("train:", X_train.shape, y_train.shape)
print("valid:", X_valid.shape, y_valid.shape)
print("test:", X_test.shape, y_test.shape)
print("train_end:", train_end)
print("valid_end:", valid_end)


## Baseline and Ridge regression

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge

def regression_metrics(y_true, y_pred, name="model"):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: RMSE={rmse:.4f} | MAE={mae:.4f} | R2={r2:.4f}")

baseline_pred = X_test["engagement_index_draft"].values
regression_metrics(y_test, baseline_pred, name="baseline_current_ei")

ridge = Ridge(alpha=1.0, random_state=SEED)
ridge.fit(X_train, y_train)

ridge_pred = ridge.predict(X_test)
regression_metrics(y_test, ridge_pred, name="ridge")

## CatBoost regression

In [ ]:
from catboost import CatBoostRegressor

cat_model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=SEED,
    verbose=100
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

cat_pred = cat_model.predict(X_test)
regression_metrics(y_test, cat_pred, name="catboost_regressor")

## CatBoost feature importance

In [ ]:
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": cat_model.get_feature_importance()
}).sort_values("importance", ascending=False)

print(feature_importance.to_string(index=False))

## EI-drop logistic classifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report

df_cls_ml = df_ml_model.dropna(subset=["target_ei_drop_flag"]).copy()
df_cls_ml["target_ei_drop_flag"] = df_cls_ml["target_ei_drop_flag"].astype(int)

for col in feature_cols:
    df_cls_ml[col] = df_cls_ml[col].replace([np.inf, -np.inf], np.nan)
    df_cls_ml[col] = df_cls_ml[col].fillna(df_cls_ml[col].median())

train_end = df_cls_ml["business_wk"].quantile(0.70)
valid_end = df_cls_ml["business_wk"].quantile(0.85)

train_mask = df_cls_ml["business_wk"] <= train_end
valid_mask = (df_cls_ml["business_wk"] > train_end) & (df_cls_ml["business_wk"] <= valid_end)
test_mask  = df_cls_ml["business_wk"] > valid_end

X_train_cls = df_cls_ml.loc[train_mask, feature_cols]
y_train_cls = df_cls_ml.loc[train_mask, "target_ei_drop_flag"]

X_test_cls = df_cls_ml.loc[test_mask, feature_cols]
y_test_cls = df_cls_ml.loc[test_mask, "target_ei_drop_flag"]

logreg = LogisticRegression(
    solver="lbfgs",
    max_iter=5000,
    class_weight="balanced",
    random_state=SEED
)

logreg.fit(X_train_cls, y_train_cls)

proba = logreg.predict_proba(X_test_cls)[:, 1]
pred = (proba >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test_cls, proba))
print("PR-AUC:", average_precision_score(y_test_cls, proba))
print("F1:", f1_score(y_test_cls, pred))
print(classification_report(y_test_cls, pred))

## EI-drop CatBoost classifier

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report
import numpy as np
import pandas as pd

df_cls_ml = df_ml_model.copy()

df_cls_ml = df_cls_ml.dropna(subset=["target_ei_drop_flag"]).copy()

for col in feature_cols:
    df_cls_ml[col] = pd.to_numeric(df_cls_ml[col], errors="coerce")
    df_cls_ml[col] = df_cls_ml[col].replace([np.inf, -np.inf], np.nan)
    df_cls_ml[col] = df_cls_ml[col].fillna(df_cls_ml[col].median())

df_cls_ml["business_wk"] = pd.to_datetime(df_cls_ml["business_wk"], errors="coerce")
df_cls_ml = df_cls_ml.dropna(subset=["business_wk"]).copy()

train_end = df_cls_ml["business_wk"].quantile(0.70)
valid_end = df_cls_ml["business_wk"].quantile(0.85)

train_mask = df_cls_ml["business_wk"] <= train_end
valid_mask = (df_cls_ml["business_wk"] > train_end) & (df_cls_ml["business_wk"] <= valid_end)
test_mask  = df_cls_ml["business_wk"] > valid_end

X_train_cls = df_cls_ml.loc[train_mask, feature_cols].copy()
y_train_cls = df_cls_ml.loc[train_mask, "target_ei_drop_flag"].copy()

X_valid_cls = df_cls_ml.loc[valid_mask, feature_cols].copy()
y_valid_cls = df_cls_ml.loc[valid_mask, "target_ei_drop_flag"].copy()

X_test_cls = df_cls_ml.loc[test_mask, feature_cols].copy()
y_test_cls = df_cls_ml.loc[test_mask, "target_ei_drop_flag"].copy()

print("train:", X_train_cls.shape, y_train_cls.shape)
print("valid:", X_valid_cls.shape, y_valid_cls.shape)
print("test:", X_test_cls.shape, y_test_cls.shape)

cat_cls = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=SEED,
    verbose=100
)

cat_cls.fit(
    X_train_cls, y_train_cls,
    eval_set=(X_valid_cls, y_valid_cls),
    use_best_model=True
)

proba_cat = cat_cls.predict_proba(X_test_cls)[:, 1]
pred_cat = (proba_cat >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test_cls, proba_cat))
print("PR-AUC:", average_precision_score(y_test_cls, proba_cat))
print("F1:", f1_score(y_test_cls, pred_cat))
print(classification_report(y_test_cls, pred_cat))

## Threshold tuning for EI-drop classifier

In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score

proba_valid = cat_cls.predict_proba(X_valid_cls)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_valid_cls, proba_valid)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)

best_idx = f1_scores.argmax()
best_threshold = thresholds[max(best_idx - 1, 0)] if best_idx < len(thresholds) else 0.5

print("best threshold on valid:", best_threshold)
print("best precision:", precisions[best_idx])
print("best recall:", recalls[best_idx])
print("best f1:", f1_scores[best_idx])

proba_cat = cat_cls.predict_proba(X_test_cls)[:, 1]
pred_cat_best = (proba_cat >= best_threshold).astype(int)

print(classification_report(y_test_cls, pred_cat_best))

## Final analytical report block

## Additional PV gap checks

In [ ]:
print(na_stats.head(20).to_string())
print(df.shape)
df_model.groupby("engagement_group_3")[["pv_next_wk", "pv_50_next_wk"]].mean()
df_model.groupby("engagement_group_5")[["pv_next_wk", "pv_50_next_wk"]].mean()

df_model = df_model.copy()

df_model["business_wk"] = pd.to_datetime(df_model["business_wk"], errors="coerce")

df_model = df_model.sort_values(["delivery_agent_rk", "business_wk"]).copy()

df_model["next_business_wk"] = (
    df_model.groupby("delivery_agent_rk")["business_wk"].shift(-1)
)

df_model["week_gap_days"] = (
    df_model["next_business_wk"] - df_model["business_wk"]
).dt.days

print("PV week gap distribution:")
print(df_model["week_gap_days"].value_counts(dropna=False).sort_index())

print(df_model["week_gap_days"].value_counts(dropna=False).sort_index())

## Feature importance chart

In [ ]:
top_n = 15

fi_plot = (
    feature_importance
    .sort_values("importance", ascending=True)
    .tail(top_n)
)

plt.figure(figsize=(10, 6))
plt.barh(fi_plot["feature"], fi_plot["importance"])
plt.title("Top CatBoost feature importance for next EI prediction")
plt.xlabel("importance")
plt.ylabel("feature")
plt.tight_layout()
plt.show()

## Final analytical report helpers

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report
)

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 300)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

def print_header(title):
    print("\n")
    print("=" * 120)
    print(title)
    print("=" * 120)

def print_subheader(title):
    print("\n" + "-" * 120)
    print(title)
    print("-" * 120)

def exists(name):
    return name in globals()

def get(name, default=None):
    return globals().get(name, default)

def safe_print_df(obj, name=None, max_rows=None):
    if name:
        print(f"\n{name}:")
    try:
        if isinstance(obj, pd.DataFrame):
            if max_rows is not None:
                print(obj.head(max_rows).to_string())
            else:
                print(obj.to_string())
        elif isinstance(obj, pd.Series):
            if max_rows is not None:
                print(obj.head(max_rows).to_string())
            else:
                print(obj.to_string())
        else:
            print(obj)
    except Exception as e:
        print(f"FAILED TO PRINT {name if name else ''}: {repr(e)}")

def coef_table(model):
    out = pd.DataFrame({
        "coef": model.params,
        "std_err": model.bse,
        "p_value": model.pvalues,
        "t_or_z": model.tvalues
    })
    return out

def regression_metrics_table(y_true, y_pred, name):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    if mask.sum() == 0:
        return pd.DataFrame([{
            "model": name,
            "rmse": np.nan,
            "mae": np.nan,
            "r2": np.nan,
            "n": 0
        }])

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    mse = mean_squared_error(y_true, y_pred)
    return pd.DataFrame([{
        "model": name,
        "rmse": np.sqrt(mse),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
        "n": len(y_true)
    }])

def classification_metrics_print(y_true, proba=None, pred=None, name="model"):
    print_subheader(f"Classification metrics: {name}")

    y_true = pd.Series(y_true).astype(float)

    if proba is not None:
        proba = np.asarray(proba)
        mask = y_true.notna().values & np.isfinite(proba)
        if mask.sum() > 0:
            print("ROC-AUC:", roc_auc_score(y_true[mask], proba[mask]))
            print("PR-AUC:", average_precision_score(y_true[mask], proba[mask]))
        else:
            print("ROC-AUC / PR-AUC: skipped, no valid proba")

    if pred is not None:
        pred = np.asarray(pred)
        mask = y_true.notna().values & pd.Series(pred).notna().values
        if mask.sum() > 0:
            print("F1:", f1_score(y_true[mask], pred[mask]))
            print(classification_report(y_true[mask], pred[mask]))
        else:
            print("F1 / report: skipped, no valid pred")


## 1. Dataset Validation

In [ ]:
print_header("1. DATASET VALIDATION")

if exists("df"):
    df_report = df.copy()

    if "business_wk" in df_report.columns:
        df_report["business_wk"] = pd.to_datetime(df_report["business_wk"], errors="coerce")

    print_subheader("1.1 Shape and main structure")
    print("df.shape:", df_report.shape)

    if "delivery_agent_rk" in df_report.columns:
        print("unique employees:", df_report["delivery_agent_rk"].nunique())

    if "business_wk" in df_report.columns:
        print("unique weeks:", df_report["business_wk"].nunique())
        print("min business_wk:", df_report["business_wk"].min())
        print("max business_wk:", df_report["business_wk"].max())

    print_subheader("1.2 Duplicates by employee-week")
    if {"delivery_agent_rk", "business_wk"}.issubset(df_report.columns):
        dup_cnt = df_report.duplicated(subset=["delivery_agent_rk", "business_wk"]).sum()
        print("dup_cnt:", dup_cnt)
    else:
        print("SKIPPED: delivery_agent_rk/business_wk not found")

    print_subheader("1.3 Weeks per employee")
    if {"delivery_agent_rk", "business_wk"}.issubset(df_report.columns):
        emp_weeks = (
            df_report.groupby("delivery_agent_rk")["business_wk"]
            .nunique()
            .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
        )
        safe_print_df(emp_weeks, "weeks per employee")
    else:
        print("SKIPPED")

    print_subheader("1.4 Missing values TOP-20")
    na_stats_report = pd.DataFrame({
        "na_cnt": df_report.isna().sum(),
        "na_share": df_report.isna().mean().round(6)
    }).sort_values(["na_share", "na_cnt"], ascending=False)

    safe_print_df(na_stats_report.head(20), "na_stats.head(20)")

    print_subheader("1.5 Logical sanity checks")
    checks = {}

    def add_check(name, expr):
        try:
            checks[name] = int(expr.sum())
        except Exception:
            checks[name] = "SKIPPED"

    if {"held_meetings_amt", "scheduled_meetings_amt"}.issubset(df_report.columns):
        add_check("held_gt_scheduled", df_report["held_meetings_amt"] > df_report["scheduled_meetings_amt"])
    else:
        checks["held_gt_scheduled"] = "SKIPPED"

    if {"success_meetings_amt", "held_meetings_amt"}.issubset(df_report.columns):
        add_check("success_gt_held", df_report["success_meetings_amt"] > df_report["held_meetings_amt"])
    else:
        checks["success_gt_held"] = "SKIPPED"

    if "held_from_scheduled_ratio" in df_report.columns:
        add_check("held_ratio_gt_1", df_report["held_from_scheduled_ratio"] > 1)
    else:
        checks["held_ratio_gt_1"] = "SKIPPED"

    if "success_from_held_ratio" in df_report.columns:
        add_check("success_ratio_gt_1", df_report["success_from_held_ratio"] > 1)
    else:
        checks["success_ratio_gt_1"] = "SKIPPED"

    if "avg_offer_voice_percent" in df_report.columns:
        add_check("voice_percent_gt_100", df_report["avg_offer_voice_percent"] > 100)
        add_check("voice_percent_lt_0", df_report["avg_offer_voice_percent"] < 0)
    else:
        checks["voice_percent_gt_100"] = "SKIPPED"
        checks["voice_percent_lt_0"] = "SKIPPED"

    if "held_meetings_amt" in df_report.columns:
        add_check("negative_meetings", df_report["held_meetings_amt"] < 0)
    else:
        checks["negative_meetings"] = "SKIPPED"

    if "qc_error_weight_sum" in df_report.columns:
        add_check("negative_qc", df_report["qc_error_weight_sum"] < 0)
    else:
        checks["negative_qc"] = "SKIPPED"

    safe_print_df(pd.Series(checks), "sanity checks")

    print_subheader("1.6 Weekly dynamics summary")
    if {"business_wk", "delivery_agent_rk"}.issubset(df_report.columns):
        agg_dict = {
            "employees_cnt": ("delivery_agent_rk", "nunique"),
            "rows_cnt": ("delivery_agent_rk", "size")
        }

        if "held_meetings_amt" in df_report.columns:
            agg_dict["held_meetings_amt"] = ("held_meetings_amt", "sum")
        if "scheduled_meetings_amt" in df_report.columns:
            agg_dict["scheduled_meetings_amt"] = ("scheduled_meetings_amt", "sum")
        if "qc_error_cnt" in df_report.columns:
            agg_dict["qc_error_cnt"] = ("qc_error_cnt", "sum")

        wk_stats_report = (
            df_report.groupby("business_wk")
            .agg(**agg_dict)
            .reset_index()
            .sort_values("business_wk")
        )

        safe_print_df(wk_stats_report.head(10), "wk_stats head(10)")
        safe_print_df(wk_stats_report.tail(10), "wk_stats tail(10)")
    else:
        print("SKIPPED")

else:
    print("SKIPPED: df not found")


## 2. Outliers And Correlations

In [ ]:
print_header("2. OUTLIERS AND CORRELATIONS")

if exists("df"):
    df_report = df.copy()

    eda_cols_default = [
        "meetings_amt",
        "scheduled_meetings_amt",
        "held_meetings_amt",
        "success_meetings_amt",
        "held_from_scheduled_ratio",
        "success_from_held_ratio",
        "avg_tasks_on_meeting_amt",
        "tasks_per_held_meeting_amt",
        "avg_offer_cnt_per_meeting_amt",
        "avg_offer_voice_percent",
        "avg_voice_score",
        "voiced_meeting_ratio",
        "avg_schedule_meeting_coverage_ratio",
        "avg_schedule_span_coverage_ratio",
        "held_meeting_days_cnt",
        "held_days_from_region_work_days_ratio",
        "qc_error_cnt",
        "qc_error_days_cnt",
        "qc_error_weight_sum",
        "qc_error_cnt_per_held_meeting",
        "qc_error_weight_sum_per_held_meeting"
    ]

    eda_cols_report = [c for c in eda_cols_default if c in df_report.columns]

    print_subheader("2.1 EDA columns used")
    print(eda_cols_report)

    print_subheader("2.2 Outlier stats by IQR rule TOP-20")
    outlier_stats_report = []

    for col in eda_cols_report:
        s = pd.to_numeric(df_report[col], errors="coerce").dropna()
        if len(s) == 0:
            continue

        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outliers = ((df_report[col] < lower) | (df_report[col] > upper)).sum()

        outlier_stats_report.append({
            "column": col,
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_cnt": int(outliers),
            "outlier_share": round(outliers / len(df_report), 6)
        })

    if len(outlier_stats_report) > 0:
        outlier_stats_report = (
            pd.DataFrame(outlier_stats_report)
            .sort_values("outlier_share", ascending=False)
        )
        safe_print_df(outlier_stats_report.head(20), "outlier_stats.head(20)")
    else:
        print("SKIPPED")

    print_subheader("2.3 Correlation pairs TOP-30")
    try:
        corr = df_report[eda_cols_report].corr()

        corr_pairs_report = (
            corr.where(~np.eye(corr.shape[0], dtype=bool))
            .stack()
            .reset_index()
        )
        corr_pairs_report.columns = ["feature_1", "feature_2", "corr"]
        corr_pairs_report["abs_corr"] = corr_pairs_report["corr"].abs()

        corr_pairs_report = (
            corr_pairs_report
            .sort_values("abs_corr", ascending=False)
            .drop_duplicates(subset=["abs_corr", "corr"])
        )

        safe_print_df(corr_pairs_report.head(30), "corr_pairs.head(30)")
    except Exception as e:
        print("SKIPPED:", repr(e))

    print_subheader("2.4 Candidate engagement feature correlation matrix")
    engagement_candidate_cols = [
        "held_from_scheduled_ratio",
        "tasks_per_held_meeting_amt",
        "avg_offer_voice_percent",
        "avg_voice_score",
        "voiced_meeting_ratio",
        "avg_schedule_meeting_coverage_ratio",
        "avg_schedule_span_coverage_ratio",
        "qc_error_cnt_per_held_meeting",
        "qc_error_weight_sum_per_held_meeting"
    ]

    engagement_candidate_cols = [c for c in engagement_candidate_cols if c in df_report.columns]

    try:
        cand_corr_report = df_report[engagement_candidate_cols].corr()
        safe_print_df(cand_corr_report, "candidate engagement features correlation")
    except Exception as e:
        print("SKIPPED:", repr(e))

else:
    print("SKIPPED: df not found")


## 3. Engagement Index Validation

In [ ]:
print_header("3. ENGAGEMENT INDEX VALIDATION")

if exists("df_idx"):
    df_idx_report = df_idx.copy()

    print_subheader("3.1 df_idx shape")
    print("df_idx.shape:", df_idx_report.shape)

    print_subheader("3.2 Engagement index components control")

    component_cols = [
        "held_from_scheduled_ratio_z",
        "tasks_per_held_meeting_amt_z",
        "avg_offer_voice_percent_z",
        "avg_voice_score_z",
        "voiced_meeting_ratio_z",
        "voice_component_z",
        "qc_error_weight_sum_per_held_meeting_z",
        "engagement_index_draft"
    ]

    component_cols = [c for c in component_cols if c in df_idx_report.columns]

    control_rows = []

    for col in component_cols:
        s = pd.to_numeric(df_idx_report[col], errors="coerce")
        control_rows.append({
            "feature": col,
            "na_cnt": int(s.isna().sum()),
            "inf_cnt": int(np.isinf(s.dropna()).sum()),
            "mean": s.mean(),
            "std": s.std(),
            "min": s.min(),
            "p50": s.median(),
            "p95": s.quantile(0.95),
            "max": s.max()
        })

    if len(control_rows) > 0:
        control_df = pd.DataFrame(control_rows)
        safe_print_df(control_df, "EI component control")
    else:
        print("SKIPPED: component columns not found")

    print_subheader("3.3 Engagement group summary")

    try:
        if "engagement_group_3" not in df_idx_report.columns:
            df_idx_report["engagement_group_3"] = pd.qcut(
                df_idx_report["engagement_index_draft"],
                q=3,
                labels=["low", "mid", "high"]
            )

        summary_cols = [
            "held_from_scheduled_ratio",
            "tasks_per_held_meeting_amt",
            "avg_offer_voice_percent",
            "avg_voice_score",
            "voiced_meeting_ratio",
            "voice_component_z",
            "qc_error_weight_sum_per_held_meeting",
            "engagement_index_draft"
        ]

        summary_cols = [c for c in summary_cols if c in df_idx_report.columns]

        group_summary_report = (
            df_idx_report.groupby("engagement_group_3")[summary_cols]
            .agg(["mean", "median"])
        )

        safe_print_df(group_summary_report, "engagement_group_3 summary")
    except Exception as e:
        print("SKIPPED:", repr(e))

    print_subheader("3.4 Engagement index descriptive statistics")
    if "engagement_index_draft" in df_idx_report.columns:
        ei_desc = df_idx_report["engagement_index_draft"].describe(
            percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
        )
        safe_print_df(ei_desc, "engagement_index_draft describe")
    else:
        print("SKIPPED")

else:
    print("SKIPPED: df_idx not found")


## 4. Pv Relationship Analysis

In [ ]:
print_header("4. PV RELATIONSHIP ANALYSIS")

if exists("df_model"):
    df_model_report = df_model.copy()

    if "business_wk" in df_model_report.columns:
        df_model_report["business_wk"] = pd.to_datetime(df_model_report["business_wk"], errors="coerce")

    print_subheader("4.1 df_model shape")
    print("df_model.shape:", df_model_report.shape)

    print_subheader("4.2 PV week gap distribution")

    try:
        if {"delivery_agent_rk", "business_wk"}.issubset(df_model_report.columns):
            df_model_report = df_model_report.sort_values(["delivery_agent_rk", "business_wk"]).copy()

            if "week_gap_days" not in df_model_report.columns:
                df_model_report["next_business_wk"] = (
                    df_model_report.groupby("delivery_agent_rk")["business_wk"].shift(-1)
                )
                df_model_report["week_gap_days"] = (
                    df_model_report["next_business_wk"] - df_model_report["business_wk"]
                ).dt.days

            safe_print_df(
                df_model_report["week_gap_days"].value_counts(dropna=False).sort_index(),
                "PV week_gap_days distribution"
            )
        else:
            print("SKIPPED: keys not found")
    except Exception as e:
        print("SKIPPED:", repr(e))

    print_subheader("4.3 PV correlations")

    try:
        if "pv_next_wk" in df_model_report.columns and "pv_next_wk_log1p" not in df_model_report.columns:
            df_model_report["pv_next_wk_log1p"] = np.log1p(
                pd.to_numeric(df_model_report["pv_next_wk"], errors="coerce").clip(lower=0)
            )

        if "pv_50_next_wk" in df_model_report.columns and "pv_50_next_wk_log1p" not in df_model_report.columns:
            df_model_report["pv_50_next_wk_log1p"] = np.log1p(
                pd.to_numeric(df_model_report["pv_50_next_wk"], errors="coerce").clip(lower=0)
            )

        corr_cols = [
            "engagement_index_draft",
            "pv_next_wk",
            "pv_50_next_wk",
            "pv_for_pi_next_wk",
            "pv_next_wk_log1p",
            "pv_50_next_wk_log1p"
        ]

        corr_cols = [c for c in corr_cols if c in df_model_report.columns]

        safe_print_df(
            df_model_report[corr_cols].corr().round(6),
            "Pearson correlation"
        )

        safe_print_df(
            df_model_report[corr_cols].corr(method="spearman").round(6),
            "Spearman correlation"
        )
    except Exception as e:
        print("SKIPPED:", repr(e))

    print_subheader("4.4 PV by engagement_group_3")

    try:
        if "engagement_group_3" not in df_model_report.columns:
            df_model_report["engagement_group_3"] = pd.qcut(
                df_model_report["engagement_index_draft"],
                q=3,
                labels=["low", "mid", "high"]
            )

        pv_group_cols = [c for c in ["pv_next_wk", "pv_50_next_wk"] if c in df_model_report.columns]

        group3_pv = (
            df_model_report.groupby("engagement_group_3")[pv_group_cols]
            .mean()
        )

        safe_print_df(group3_pv, "PV mean by engagement_group_3")
    except Exception as e:
        print("SKIPPED:", repr(e))

    print_subheader("4.5 PV by engagement_group_5")

    try:
        if "engagement_group_5" not in df_model_report.columns:
            df_model_report["engagement_group_5"] = pd.qcut(
                df_model_report["engagement_index_draft"],
                q=5,
                labels=["q1", "q2", "q3", "q4", "q5"]
            )

        pv_group_cols = [c for c in ["pv_next_wk", "pv_50_next_wk"] if c in df_model_report.columns]

        group5_pv = (
            df_model_report.groupby("engagement_group_5")[pv_group_cols]
            .mean()
        )

        safe_print_df(group5_pv, "PV mean by engagement_group_5")
    except Exception as e:
        print("SKIPPED:", repr(e))

else:
    print("SKIPPED: df_model not found")


## 5. Ols Models

In [ ]:
print_header("5. OLS MODELS")

model_names = [
    "model_1",
    "model_2",
    "model_3",
    "model_4",
    "model_base",
    "model_3_cluster"
]

for model_name in model_names:
    print_subheader(f"5.X {model_name}")

    if exists(model_name):
        model_obj = get(model_name)
        try:
            safe_print_df(coef_table(model_obj), f"{model_name} coef table")
            if hasattr(model_obj, "rsquared"):
                print("R2:", model_obj.rsquared)
            if hasattr(model_obj, "rsquared_adj"):
                print("Adj R2:", model_obj.rsquared_adj)
            if hasattr(model_obj, "nobs"):
                print("nobs:", model_obj.nobs)
        except Exception as e:
            print("FAILED:", repr(e))
    else:
        print("SKIPPED: model not found")

print_subheader("5.7 DELTA R2")

if exists("model_base") and exists("model_3"):
    try:
        print("BASE MODEL R2:", model_base.rsquared)
        print("MODEL 3 R2:", model_3.rsquared)
        print("DELTA R2:", model_3.rsquared - model_base.rsquared)
    except Exception as e:
        print("FAILED:", repr(e))
else:
    print("SKIPPED: model_base or model_3 not found")


## 6. Segmentation And Scenario Analysis

In [ ]:
print_header("6. SEGMENTATION AND SCENARIO ANALYSIS")

print_subheader("6.1 Segment summary")

if exists("segment_summary_named"):
    safe_print_df(segment_summary_named, "segment_summary_named")
elif exists("segment_summary"):
    safe_print_df(segment_summary, "segment_summary")
else:
    print("SKIPPED: segment_summary not found")

print_subheader("6.2 Segment share")

if exists("segment_share"):
    safe_print_df(segment_share, "segment_share")
else:
    print("SKIPPED: segment_share not found")

print_subheader("6.3 Segment profiles")

if exists("segment_profiles"):
    safe_print_df(segment_profiles, "segment_profiles")
else:
    print("SKIPPED: segment_profiles not found")

print_subheader("6.4 Scenario grid")

if exists("scenario_grid"):
    safe_print_df(scenario_grid, "scenario_grid")
else:
    print("SKIPPED: scenario_grid not found")

print_subheader("6.5 Scenario scalar outputs")

scenario_scalar_names = [
    "total_observed_pv",
    "company_weekly_pv",
    "company_avg_pv_per_row",
    "expected_observed_pv_after",
    "expected_company_pv_after",
    "expected_pv_difference",
    "expected_loss",
    "seg_total_pv",
    "seg_expected_loss",
    "seg_top_total_pv",
    "seg_top_expected_loss",
    "seg_10pct_total_pv",
    "seg_10pct_expected_loss"
]

for name in scenario_scalar_names:
    if exists(name):
        print(f"{name}: {get(name)}")


## 7. Ml Regression

In [ ]:
print_header("7. ML REGRESSION")

print_subheader("7.1 ML dataset and split shapes")

shape_vars = [
    ("X_train", "y_train"),
    ("X_valid", "y_valid"),
    ("X_test", "y_test")
]

for x_name, y_name in shape_vars:
    if exists(x_name) and exists(y_name):
        print(f"{x_name}: {get(x_name).shape} | {y_name}: {get(y_name).shape}")
    else:
        print(f"{x_name}/{y_name}: SKIPPED")

print_subheader("7.2 Regression metrics")

reg_metrics = []

if exists("y_test") and exists("baseline_pred"):
    reg_metrics.append(regression_metrics_table(y_test, baseline_pred, "baseline_current_ei"))

if exists("y_test") and exists("ridge_pred"):
    reg_metrics.append(regression_metrics_table(y_test, ridge_pred, "ridge"))

if exists("y_test") and exists("cat_pred"):
    reg_metrics.append(regression_metrics_table(y_test, cat_pred, "catboost_regressor"))

if len(reg_metrics) > 0:
    reg_metrics_df = pd.concat(reg_metrics, ignore_index=True)
    safe_print_df(reg_metrics_df, "regression_metrics")
else:
    print("SKIPPED: predictions not found")

print_subheader("7.3 Feature importance TOP-20")

if exists("feature_importance"):
    try:
        safe_print_df(feature_importance.head(20), "feature_importance.head(20)")
    except Exception as e:
        print("FAILED:", repr(e))
else:
    print("SKIPPED: feature_importance not found")


## 8. Ml Classification: Ei Drop

In [ ]:
print_header("8. ML CLASSIFICATION: EI DROP")

print_subheader("8.1 Classification target distribution")

if exists("df_cls_ml") and "target_ei_drop_flag" in df_cls_ml.columns:
    safe_print_df(
        df_cls_ml["target_ei_drop_flag"].value_counts(dropna=False),
        "target_ei_drop_flag value_counts"
    )
    safe_print_df(
        df_cls_ml["target_ei_drop_flag"].value_counts(normalize=True, dropna=False),
        "target_ei_drop_flag value_counts normalize"
    )
elif exists("df_ml") and "target_ei_drop_flag" in df_ml.columns:
    safe_print_df(
        df_ml["target_ei_drop_flag"].value_counts(dropna=False),
        "target_ei_drop_flag value_counts"
    )
    safe_print_df(
        df_ml["target_ei_drop_flag"].value_counts(normalize=True, dropna=False),
        "target_ei_drop_flag value_counts normalize"
    )
else:
    print("SKIPPED: target_ei_drop_flag not found")

print_subheader("8.2 Classification split shapes")

cls_shape_vars = [
    ("X_train_cls", "y_train_cls"),
    ("X_valid_cls", "y_valid_cls"),
    ("X_test_cls", "y_test_cls")
]

for x_name, y_name in cls_shape_vars:
    if exists(x_name) and exists(y_name):
        print(f"{x_name}: {get(x_name).shape} | {y_name}: {get(y_name).shape}")
    else:
        print(f"{x_name}/{y_name}: SKIPPED")

print_subheader("8.3 Logistic Regression classification metrics")

if exists("y_test_cls") and exists("proba"):
    pred_for_logreg = get("pred", None)
    classification_metrics_print(y_test_cls, proba=proba, pred=pred_for_logreg, name="logistic_regression")
else:
    print("SKIPPED: y_test_cls/proba not found")

print_subheader("8.4 CatBoost classification metrics with default threshold")

if exists("y_test_cls") and exists("proba_cat"):
    pred_for_cat = get("pred_cat", None)
    classification_metrics_print(y_test_cls, proba=proba_cat, pred=pred_for_cat, name="catboost_default_threshold")
else:
    print("SKIPPED: y_test_cls/proba_cat not found")

print_subheader("8.5 CatBoost classification metrics with tuned threshold")

if exists("best_threshold"):
    print("best_threshold:", best_threshold)
if exists("best_precision"):
    print("best_precision:", best_precision)
if exists("best_recall"):
    print("best_recall:", best_recall)
if exists("best_f1"):
    print("best_f1:", best_f1)

if exists("y_test_cls") and exists("proba_cat") and exists("best_threshold"):
    pred_cat_best_report = (proba_cat >= best_threshold).astype(int)
    classification_metrics_print(
        y_test_cls,
        proba=proba_cat,
        pred=pred_cat_best_report,
        name="catboost_tuned_threshold_on_test"
    )
elif exists("y_test_cls") and exists("pred_cat_best"):
    classification_metrics_print(
        y_test_cls,
        proba=get("proba_cat", None),
        pred=pred_cat_best,
        name="catboost_tuned_threshold_on_test"
    )
else:
    print("SKIPPED: tuned threshold outputs not found")


## 9. Ml Week Gap Distribution

In [ ]:
print_header("9. ML WEEK GAP DISTRIBUTION")

if exists("df_ml"):
    df_ml_report = df_ml.copy()

    try:
        df_ml_report["business_wk"] = pd.to_datetime(df_ml_report["business_wk"], errors="coerce")
        df_ml_report = df_ml_report.sort_values(["delivery_agent_rk", "business_wk"]).copy()

        if "week_gap_days" not in df_ml_report.columns:
            df_ml_report["next_business_wk"] = (
                df_ml_report.groupby("delivery_agent_rk")["business_wk"].shift(-1)
            )
            df_ml_report["week_gap_days"] = (
                df_ml_report["next_business_wk"] - df_ml_report["business_wk"]
            ).dt.days

        safe_print_df(
            df_ml_report["week_gap_days"].value_counts(dropna=False).sort_index(),
            "EI/ML week_gap_days distribution"
        )
    except Exception as e:
        print("SKIPPED:", repr(e))
else:
    print("SKIPPED: df_ml not found")


## Final Report Output Block Finished

In [ ]:
print_header("FINAL REPORT OUTPUT BLOCK FINISHED")
